In [0]:
import zipfile
import json
import os
import pytz
from pyspark.sql.types import StructType, IntegerType, DoubleType, LongType
from pyspark.sql import functions as F, SparkSession, DataFrame
from pyspark.sql.utils import AnalysisException
from datetime import datetime
from functools import reduce
from delta.tables import DeltaTable

In [0]:
### Funções de Tratamento Base

def clean_and_fill(df, columns, replacement="NÃO INFORMADO"):
    for column in columns:
        df = df.withColumn(
            column, 
            F.when(F.trim(F.col(column)) == "", None).otherwise(F.col(column))
        )
    return df.fillna(replacement, subset=columns)

def lower_and_upper(df, columns):
    for column in columns:
        df = df.withColumn(column, F.lower(F.col(column)))
    df = df.toDF(*[c.upper() for c in df.columns])
    return df

def tratamento(df, colunas):
    df = clean_and_fill(df, colunas)
    df = lower_and_upper(df, colunas)
    return df

In [0]:
### Configuração de Autenticação
STORAGE = "stgbbb"
spark.conf.set(
    f"fs.azure.account.key.{STORAGE}.dfs.core.windows.net",
    "xzhcoxUXRj+GjVy8wOc3wladWad/Dm+OgQ5Lpj6U0RKd8K0+n18AJz12D3FmA/LTGP8SsbqSRybt+AStxQY00w=="
)